# A.5. Reservoir and training sensitivity

Vary training length, reservoir size and spectral radius one at a time. Each configuration uses 20 paired ESNs, 30 histories per model and five evolution steps.

Data and completed models are reused. Missing data and model results are computed.


In [ ]:
import gc
import hashlib
import json
import os
import pickle
import tempfile
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
%run ./time_Lorenz_data_6time.ipynb
%run ./ESN.ipynb
%run ./Reconstruct.ipynb
N_x = 4
N_washout = 100
N_test = 2000
N_net = 20
N_rep = 30
N_evo = 5
window = 6
window_length = 100
test_window = 6
lag_list = np.arange(test_window, dtype=int) * window_length
training_lengths = np.array([2000, 5000, 10000, 15000, 20000], dtype=int)
reservoir_sizes = np.array([3600, 5400, 7200, 9000], dtype=int)
spectral_radii = np.array([0.5, 0.7, 0.9, 1.1], dtype=float)
DEFAULT_TRAIN = 10000
DEFAULT_UNITS = 7200
DEFAULT_RHO = 0.9
SIGMA_IN = 0.5
TIKHONOV = 0.0001
J_THRESHOLD = 0.01
sigma_dyn = 1e-06
sigma_obs = 1e-06
c = 0.3
dt = 0.01
density = 0.2
BURN_IN = 2000
NETWORK_SEED = 99
DATA_SEED = 199
OFFSET_SEED = 299
RESERVOIR_SEED = 499
TRUE_EDGE = (3, 0, 1)
RETRY_FAILED = False
PIPELINE_VERSION = 'further_study_4_v1'
N_dim = 3 * N_x
N_dim_segments = test_window * N_dim
max_lag = int(lag_list.max())
TRAIN_STOP = N_washout + int(training_lengths.max())
TEST_START = TRAIN_STOP + 1 + max_lag
EMBEDDED_LENGTH = TEST_START + N_test
HISTORY_LENGTH = window * window_length
RAW_ORIGIN = HISTORY_LENGTH + BURN_IN + max_lag
baseline_condition = (DEFAULT_TRAIN, DEFAULT_UNITS, DEFAULT_RHO)
sweep_conditions = {'training_length': [(int(length), DEFAULT_UNITS, DEFAULT_RHO) for length in training_lengths], 'reservoir_size': [(DEFAULT_TRAIN, int(size), DEFAULT_RHO) for size in reservoir_sizes], 'spectral_radius': [(DEFAULT_TRAIN, DEFAULT_UNITS, float(radius)) for radius in spectral_radii]}
conditions = list(dict.fromkeys([baseline_condition] + [condition for sweep in sweep_conditions.values() for condition in sweep]))

def notebook_source(cell):
    return ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
dependency_hashes = {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in ('time_Lorenz_data_6time.ipynb', 'ESN.ipynb', 'Reconstruct.ipynb')}
own_notebook = json.loads(Path('A.5.ipynb').read_text(encoding='utf-8'))
computation_source = '\n'.join((notebook_source(own_notebook['cells'][index]) for index in (1, 3, 5)))
computation_source = '\n'.join((line for line in computation_source.splitlines() if not line.startswith('RETRY_FAILED =')))
computation_hash = hashlib.sha256(computation_source.encode('utf-8')).hexdigest()
cache_compatibility = own_notebook.get('metadata', {}).get('cache_compatibility', {})
current_computation_hash = hashlib.sha256(computation_source.encode('utf-8')).hexdigest()
compatible_computation_hash = cache_compatibility.get(current_computation_hash, current_computation_hash)
experiment_signature = {'pipeline': PIPELINE_VERSION, 'computation_hash': compatible_computation_hash, 'dependency_hashes': dependency_hashes, 'numpy_version': np.__version__, 'N_x': N_x, 'N_washout': N_washout, 'N_test': N_test, 'N_net': N_net, 'N_rep': N_rep, 'N_evo': N_evo, 'window': window, 'window_length': window_length, 'lag_list': lag_list.tolist(), 'training_lengths': training_lengths.tolist(), 'reservoir_sizes': reservoir_sizes.tolist(), 'spectral_radii': spectral_radii.tolist(), 'baseline': list(baseline_condition), 'sigma_in': SIGMA_IN, 'tikhonov': TIKHONOV, 'j_threshold': J_THRESHOLD, 'sigma_dyn': sigma_dyn, 'sigma_obs': sigma_obs, 'c': c, 'dt': dt, 'density': density, 'burn_in': BURN_IN, 'network_seed': NETWORK_SEED, 'data_seed': DATA_SEED, 'offset_seed': OFFSET_SEED, 'reservoir_seed': RESERVOIR_SEED, 'true_edge': list(TRUE_EDGE)}
signature_token = hashlib.sha256(json.dumps(experiment_signature, sort_keys=True).encode('utf-8')).hexdigest()
RESULT_DIR = Path('result') / 'A.5'
CACHE_DIR = RESULT_DIR / 'runs' / signature_token[:20]
DATA_PATH = CACHE_DIR / 'data.pkl'
RESULT_PATH = RESULT_DIR / 'A.5.pkl'
FIGURE_DIR = Path('result') / 'figure'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def save_pickle_atomic(value, path):
    path = Path(path)
    (descriptor, temporary_path) = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
    with os.fdopen(descriptor, 'wb') as stream:
        pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_path, path)


## Part 1. Load the shared data

Use one Lorenz trajectory, training-calibrated median/IQR normalization, nested training windows and a fixed held-out test block. Cached observations and history offsets are reused.


In [ ]:
if DATA_PATH.exists():
    with DATA_PATH.open('rb') as input_file:
        data_bundle = pickle.load(input_file)
    if data_bundle.get('signature_token') != signature_token:
        raise RuntimeError('The cached data signature does not match this experiment.')
else:
    network_rng = np.random.RandomState(NETWORK_SEED)
    adjacency = []
    for lag_index in range(test_window):
        layer = (network_rng.rand(N_x, N_x) < density).astype(float)
        np.fill_diagonal(layer, 0.0)
        adjacency.append(layer)
    adjacency[TRUE_EDGE[0]][TRUE_EDGE[1], TRUE_EDGE[2]] = 1.0
    A_tensor = np.stack(adjacency)
    true_edge_mask = A_tensor.astype(bool)
    candidate_mask = np.broadcast_to(~np.eye(N_x, dtype=bool), A_tensor.shape).copy()
    false_edge_mask = candidate_mask & ~true_edge_mask
    if not np.any(false_edge_mask):
        raise ValueError('The experiment requires at least one false candidate edge.')
    np.random.seed(DATA_SEED)
    heterogeneity = 2.0 * (np.random.rand(N_x) - 0.5) * 0.06
    initial_vector = np.random.rand(N_dim)
    initial_history = np.tile(initial_vector, (HISTORY_LENGTH, 1))
    settings = (N_x, dt, RAW_ORIGIN + EMBEDDED_LENGTH, HISTORY_LENGTH, heterogeneity, c, *adjacency, window_length, sigma_dyn)
    X_raw = RK4_6delay(coupled_Lorenz_6delay, initial_history, settings)
    if not np.all(np.isfinite(X_raw)):
        raise FloatingPointError('The generated Lorenz trajectory is not finite.')
    calibration_start = RAW_ORIGIN + TRAIN_STOP - int(training_lengths.min())
    calibration_stop = RAW_ORIGIN + TRAIN_STOP
    calibration_data = X_raw[calibration_start:calibration_stop]
    quartiles = np.percentile(calibration_data, [25, 50, 75], axis=0, method='midpoint')
    median = quartiles[1]
    iqr = quartiles[2] - quartiles[0]
    if np.any(iqr < 1e-12):
        raise ValueError('A calibration component has a near-zero interquartile range.')
    X_observed = (X_raw - median) / iqr
    if sigma_obs != 0:
        X_observed += np.random.normal(0.0, sigma_obs, size=X_observed.shape)
    X_segments = np.concatenate([X_observed[RAW_ORIGIN - int(delay):RAW_ORIGIN - int(delay) + EMBEDDED_LENGTH] for delay in lag_list], axis=1)
    X_test_segments = X_segments[TEST_START:TEST_START + N_test]
    offset_rng = np.random.RandomState(OFFSET_SEED)
    offset = offset_rng.randint(N_washout, N_test - N_evo, size=(N_net, N_rep))
    data_bundle = {'signature_token': signature_token, 'experiment_signature': experiment_signature, 'A_tensor': A_tensor, 'true_edge_mask': true_edge_mask, 'false_edge_mask': false_edge_mask, 'candidate_mask': candidate_mask, 'X_raw': X_raw, 'X_segments': X_segments, 'X_test_segments': X_test_segments, 'offset': offset, 'heterogeneity': heterogeneity, 'median': median, 'iqr': iqr, 'calibration_raw_bounds': (calibration_start, calibration_stop), 'raw_origin': RAW_ORIGIN, 'train_stop': TRAIN_STOP, 'test_start': TEST_START}
    save_pickle_atomic(data_bundle, DATA_PATH)
X_segments = np.asarray(data_bundle['X_segments'])
X_test_segments = np.asarray(data_bundle['X_test_segments'])
candidate_mask = np.asarray(data_bundle['candidate_mask'], dtype=bool)
true_edge_mask = np.asarray(data_bundle['true_edge_mask'], dtype=bool)
false_edge_mask = np.asarray(data_bundle['false_edge_mask'], dtype=bool)
offset = np.asarray(data_bundle['offset'], dtype=int)
if TEST_START - max_lag <= TRAIN_STOP:
    raise RuntimeError('Training targets and delayed test inputs overlap.')

def training_slices(training_length):
    start = TRAIN_STOP - int(training_length)
    washout_data = X_segments[start - N_washout:start]
    training_data = X_segments[start:TRAIN_STOP]
    target_data = X_segments[start + 1:TRAIN_STOP + 1]
    if len(washout_data) != N_washout or len(training_data) != training_length or len(target_data) != training_length:
        raise ValueError('The requested nested training slice has an invalid length.')
    return (washout_data, training_data, target_data)


## Part 2. Compute the DDI matrices

Fit only explicitly enabled missing configurations. DDI uses signed source-mean normalization and positive-history averaging. Model matrices are averaged before comparing true-edge and maximum absent-edge scores.


In [ ]:
def calculate_paired_ddi(esn, test_data, offsets):
    score_sums = np.zeros((test_window, N_x, N_x), dtype=float)
    positive_counts = np.zeros_like(score_sums, dtype=int)
    for current_offset in offsets:
        washout_data = test_data[current_offset - N_washout:current_offset]
        state = esn.open_loop(washout_data, np.zeros(esn.n_units))[-1]
        control = esn.evolve(state, N_evo)
        if not np.all(np.isfinite(control)):
            raise FloatingPointError('A control rollout is non-finite.')
        for lag_index in range(test_window):
            for effect in range(N_x):
                for cause in range(N_x):
                    if effect == cause:
                        continue
                    target_effect = 3 * effect
                    target_cause = 3 * cause + N_dim * lag_index + 1
                    source_mean = np.mean(control[:, target_cause])
                    if abs(source_mean) < J_THRESHOLD:
                        continue
                    intervention = esn.evolve_edge_removal(state, target_cause, target_effect, N_evo)
                    if not np.all(np.isfinite(intervention)):
                        raise FloatingPointError('An intervention rollout is non-finite.')
                    delta = control[:, target_effect] - intervention[:, target_effect]
                    score = np.sum(delta ** 2) / N_evo / source_mean
                    if not np.isfinite(score):
                        raise FloatingPointError('A DDI score is non-finite.')
                    if score > 0:
                        score_sums[lag_index, effect, cause] += score
                        positive_counts[lag_index, effect, cause] += 1
    matrices = np.divide(score_sums, positive_counts, out=np.zeros_like(score_sums), where=positive_counts > 0)
    return (matrices, positive_counts)
matrix_shape = (len(conditions), N_net, test_window, N_x, N_x)
DDI_all = np.full(matrix_shape, np.nan)
positive_counts_all = np.zeros(matrix_shape, dtype=int)
status_all = np.full((len(conditions), N_net), 'pending', dtype=object)
error_all = np.full(status_all.shape, '', dtype=object)
elapsed_all = np.full(status_all.shape, np.nan)

def checkpoint_path(condition, net):
    (training_length, units, radius) = condition
    return CACHE_DIR / f'train_{training_length}_units_{units}_rho_{radius:.8g}_net_{net:02d}.pkl'

def apply_checkpoint(payload, condition_index, net):
    if payload.get('signature_token') != signature_token or tuple(payload.get('condition', ())) != conditions[condition_index] or payload.get('net') != net:
        raise RuntimeError('A per-run checkpoint is incompatible with the current experiment.')
    if payload['status'] == 'ok':
        matrices = np.asarray(payload['DDI'], dtype=float)
        if matrices.shape != matrix_shape[2:] or not np.all(np.isfinite(matrices)):
            raise ValueError('A completed checkpoint contains invalid DDI matrices.')
        DDI_all[condition_index, net] = matrices
        positive_counts_all[condition_index, net] = payload['positive_counts']
    status_all[condition_index, net] = payload['status']
    error_all[condition_index, net] = payload.get('error', '')
    elapsed_all[condition_index, net] = payload['elapsed_seconds']

def result_bundle():
    return {'signature_token': signature_token, 'experiment_signature': experiment_signature, 'conditions': conditions, 'baseline_condition': baseline_condition, 'sweep_conditions': sweep_conditions, 'DDI_all': DDI_all, 'positive_counts_all': positive_counts_all, 'status': status_all, 'errors': error_all, 'elapsed_seconds': elapsed_all, 'A_tensor': data_bundle['A_tensor'], 'true_edge_mask': true_edge_mask, 'false_edge_mask': false_edge_mask, 'candidate_mask': candidate_mask, 'offset': offset, 'lag_list': lag_list, 'data_path': str(DATA_PATH)}
for (condition_index, condition) in enumerate(conditions):
    for net in range(N_net):
        path = checkpoint_path(condition, net)
        if path.exists():
            with path.open('rb') as input_file:
                payload = pickle.load(input_file)
            apply_checkpoint(payload, condition_index, net)
            if RETRY_FAILED and payload['status'] == 'failed':
                status_all[condition_index, net] = 'pending'
unit_order = list(dict.fromkeys((condition[1] for condition in conditions)))
for net in range(N_net):
    for units in unit_order:
        pending_indices = [index for (index, condition) in enumerate(conditions) if condition[1] == units and status_all[index, net] == 'pending']
        if not pending_indices:
            continue
        esn = None
        np.random.seed(RESERVOIR_SEED + net)
        esn = ESN(units, N_dim_segments, N_dim_segments, sigma_in=SIGMA_IN)
        esn.tikh = TIKHONOV
        if not np.all(np.isfinite(esn.W)):
            raise FloatingPointError('The initialized recurrent matrix is non-finite.')
        for condition_index in pending_indices:
            condition = conditions[condition_index]
            started = time.perf_counter()
            payload = {'signature_token': signature_token, 'condition': condition, 'net': net, 'status': 'failed', 'error': ''}
            if not '':
                esn.rho = float(condition[2])
                esn.train(*training_slices(condition[0]))
                if not np.all(np.isfinite(esn.Wout)):
                    raise FloatingPointError('The trained readout is non-finite.')
                (matrices, counts) = calculate_paired_ddi(esn, X_test_segments, offset[net])
                payload.update(status='ok', error='', DDI=matrices, positive_counts=counts)
            payload['elapsed_seconds'] = time.perf_counter() - started
            save_pickle_atomic(payload, checkpoint_path(condition, net))
            apply_checkpoint(payload, condition_index, net)
            save_pickle_atomic(result_bundle(), RESULT_PATH)
        del esn
        gc.collect()
save_pickle_atomic(result_bundle(), RESULT_PATH)


## Part 3. Plot

Load completed results and save three PNG figures. The x-axes are training length, reservoir size and spectral radius; the y-axis is DDI.


In [ ]:
with RESULT_PATH.open('rb') as input_file:
    saved_results = pickle.load(input_file)
if saved_results.get('signature_token') != signature_token:
    raise RuntimeError('The result signature differs from the current configuration.')
saved_status = np.asarray(saved_results['status'])
if not np.all(saved_status == 'ok'):
    unfinished = int(np.sum(saved_status != 'ok'))
    raise RuntimeError(f'{unfinished} runs are incomplete or failed. Inspect Part 2 statuses and finish the runs before plotting; failed runs are not excluded.')
DDI_all = np.asarray(saved_results['DDI_all'], dtype=float)
if not np.all(np.isfinite(DDI_all)):
    raise FloatingPointError('The saved DDI tensor contains non-finite values.')
condition_lookup = {tuple(condition): index for (index, condition) in enumerate(saved_results['conditions'])}
false_coordinates = np.argwhere(false_edge_mask)

def summarize_parameter_robustness(sweep_name, values):
    indices = [condition_lookup[tuple(condition)] for condition in saved_results['sweep_conditions'][sweep_name]]
    matrices = DDI_all[indices]
    averaged = np.mean(matrices, axis=1)
    true_values = matrices[:, :, TRUE_EDGE[0], TRUE_EDGE[1], TRUE_EDGE[2]]
    selected_false = np.argmax(averaged[:, false_edge_mask], axis=1)
    false_values = np.stack([matrices[level, :, coordinate[0], coordinate[1], coordinate[2]] for (level, coordinate) in enumerate(false_coordinates[selected_false])])
    return {'values': np.asarray(values), 'true_mean': np.mean(true_values, axis=1), 'true_std': np.std(true_values, axis=1, ddof=1), 'max_false_mean': np.mean(false_values, axis=1), 'max_false_std': np.std(false_values, axis=1, ddof=1), 'selected_false_coordinates': false_coordinates[selected_false]}
parameter_sweeps = (('training_length', training_lengths, 'Training length', 4), ('reservoir_size', reservoir_sizes, 'Reservoir size', 3), ('spectral_radius', spectral_radii, 'Spectral radius $\\rho_{\\mathrm{ESN}}$', None))
plot_summaries = {sweep_name: summarize_parameter_robustness(sweep_name, values) for (sweep_name, values, xlabel, exponent) in parameter_sweeps}
data_top = max((float(np.max(summary[f'{series}_mean'] + summary[f'{series}_std'])) for summary in plot_summaries.values() for series in ('true', 'max_false')))
scale_top = max(data_top, 1e-12)
common_ymax = MaxNLocator(nbins=5, min_n_ticks=4).tick_values(0.0, 1.1 * scale_top)[-1]

def plot_parameter_robustness(sweep_name, values, xlabel, exponent):
    summary = plot_summaries[sweep_name]
    with plt.rc_context({'font.family': 'serif', 'font.serif': ['Times New Roman'], 'font.size': 14, 'mathtext.fontset': 'stix'}):
        (figure, axes) = plt.subplots(figsize=(8.6, 6.4), facecolor='white')
        figure.subplots_adjust(left=0.14, right=0.97, bottom=0.16, top=0.94)
        series = (('true', '#0072B2', 'o', '-', 'True edge'), ('max_false', '#D55E00', 's', '--', 'Maximum false edge'))
        for (name, color, marker, linestyle, label) in series:
            mean = summary[f'{name}_mean']
            spread = summary[f'{name}_std']
            axes.plot(values, mean, color=color, marker=marker, markersize=6, linewidth=2.2, linestyle=linestyle, label=label)
            axes.fill_between(values, np.maximum(mean - spread, 0), mean + spread, color=color, alpha=0.15, linewidth=0)
        axes.set_xlabel(xlabel, fontsize=16)
        axes.set_ylabel('DDI', fontsize=16)
        axes.set_xticks(values)
        axes.margins(x=0.04)
        axes.set_ylim(0, common_ymax)
        axes.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))
        axes.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
        axes.yaxis.get_offset_text().set_fontsize(10)
        axes.tick_params(axis='both', labelsize=12, width=1.0, direction='out')
        axes.grid(False, which='both')
        for spine in axes.spines.values():
            spine.set_linewidth(1.0)
        axes.legend(loc='upper right', bbox_to_anchor=(0.98, 0.98), frameon=False, fontsize=12, handlelength=2.6, labelspacing=0.7)
        if exponent is not None:
            axes.ticklabel_format(axis='x', style='sci', scilimits=(exponent, exponent), useMathText=True)
            axes.xaxis.get_offset_text().set_fontsize(10)
        png_path = FIGURE_DIR / f'A.5_{sweep_name}.png'
        figure.savefig(png_path, dpi=600, facecolor='white')
        plt.show()
    return figure
figures = {sweep_name: plot_parameter_robustness(sweep_name, values, xlabel, exponent) for (sweep_name, values, xlabel, exponent) in parameter_sweeps}
print(FIGURE_DIR)
